In [ ]:
import pandas as pd
import numpy as np
import seaborn as sms
import matplotlib.pyplot as plt


In [ ]:
sales= pd.read_csv("sales_date.csv")


In [ ]:
sales.head()

In [ ]:
c= pd.read_csv("Customer.csv",index_col=0)

In [ ]:
c.head()

In [ ]:
df = sales.merge(
    c[['customer_id']],
    on = ['customer_id'],
    how='inner'
).sample(1000)


In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
import pandas as pd


#df['order_date'] = pd.to_datetime(df['order_date'])

# Calculate spend for each transaction
df['spend'] = df['product_count'] * df['price_after_discount']

# Aggregate at Customer + Product + Date level
customer_product_date = (
    df.groupby(['customer_id', 'product_id', 'order_date'])
      .agg(
          purchase_count=('order_id', 'nunique'),
          total_quantity=('product_count', 'sum'),
          avg_quantity=('product_count', 'mean'),
          max_quantity=('product_count', 'max'),
          quantity_std=('product_count', 'std'),
          total_spend=('spend', 'sum'),
          avg_spend=('spend', 'mean'),
          min_price=('spend', 'min'),
          max_price=('spend', 'max'),
          avg_price=('spend', 'mean'),
          price_paid_std=('spend', 'std'),
          avg_discount=('product_discount', 'mean'),
          max_discount=('product_discount', 'max'),
          discount_purchase_ratio=('product_discount', 'max'),
          discount_sensitivity_score=('product_discount', 'max'),
          
      )
      .reset_index()
)

customer_product_date.sample(5)

In [ ]:
"""
Customer x Product Feature Engineering
========================================
Builds the full feature table from the spec, grouped by (customer_id, product_id).

ASSUMED INPUT COLUMNS (rename to match your data before running):
    df:
        order_id              - unique order/transaction id
        customer_id           - customer identifier
        product_id            - product identifier
        order_date            - date of the order (string or datetime)
        product_count         - quantity purchased in that line item
        product_discount      - discount amount/% applied on that line item
        price_after_discount  - price actually paid PER UNIT (after discount)

    customer_df (optional, for Geography / tenure features):
        customer_id, Address, signup_date

If your column names differ, just edit the RENAME dict below.
"""

import pandas as pd
import numpy as np

# ------------------------------------------------------------------
# 0. LOAD + STANDARDIZE COLUMN NAMES
# ------------------------------------------------------------------
# df = pd.read_csv("orders.csv")
# customer_df = pd.read_csv("customer_details.csv")

RENAME = {
    # "old_col": "new_col"   <-- edit if your raw file uses different names
}
df = df.rename(columns=RENAME)

df['order_date'] = pd.to_datetime(df['order_date'])

# Total money the customer actually spent on that line item
df['spend'] = df['product_count'] * df['price_after_discount']

# "Reference date" = the point in time from which all recency windows
# (7d/30d/90d/365d, days_since_last_purchase, etc.) are measured.
REFERENCE_DATE = df['order_date'].max() + pd.Timedelta(days=1)

# ------------------------------------------------------------------
# 1. SMALL HELPER FUNCTIONS
# ------------------------------------------------------------------

def safe_std(s):
    """std() on a single value returns NaN -> fill with 0 (no variation)."""
    return s.std(ddof=0) if len(s) > 1 else 0.0


def mode_or_nan(s):
    """Most frequent value in a Series (used for month/day-of-week mode)."""
    m = s.mode()
    return m.iloc[0] if not m.empty else np.nan


def pct_change_ratio(recent, older):
    """Generic ratio used for the 30d-vs-90d trend features."""
    if older == 0:
        return np.nan if recent == 0 else np.inf
    return recent / older


# ------------------------------------------------------------------
# 2. VOLUME + SPEND + DISCOUNT + TIMING  (per customer_id, product_id)
# ------------------------------------------------------------------
# These are all straightforward aggregates across every transaction the
# customer ever made for that product, so a single groupby().agg() covers them.

base = (
    df.groupby(['customer_id', 'product_id'])
      .agg(
          # --- Volume ---
          purchase_count=('order_id', 'nunique'),
          total_quantity=('product_count', 'sum'),
          avg_quantity=('product_count', 'mean'),
          max_quantity=('product_count', 'max'),
          quantity_std=('product_count', safe_std),

          # --- Spend ---
          total_spend=('spend', 'sum'),
          avg_spend=('spend', 'mean'),
          min_price=('price_after_discount', 'min'),
          max_price=('price_after_discount', 'max'),
          avg_price=('price_after_discount', 'mean'),
          price_paid_std=('price_after_discount', safe_std),

          # --- Discount ---
          avg_discount=('product_discount', 'mean'),
          max_discount=('product_discount', 'max'),

          # --- Timing ---
          first_purchase_date=('order_date', 'min'),
          last_purchase_date=('order_date', 'max'),
      )
      .reset_index()
)

# % of purchases made with any discount at all (discount > 0)
discount_ratio = (
    df.assign(had_discount=df['product_discount'] > 0)
      .groupby(['customer_id', 'product_id'])['had_discount']
      .mean()
      .rename('discount_purchase_ratio')
      .reset_index()
)
base = base.merge(discount_ratio, on=['customer_id', 'product_id'], how='left')


# discount_sensitivity_score: correlation between discount offered and qty bought
# price_elasticity_proxy: %change in qty vs %change in price paid (line-to-line)
def discount_and_elasticity(g):
    g = g.sort_values('order_date')
    # correlation needs >=2 points and some variance, else undefined
    if len(g) > 1 and g['product_discount'].std() > 0 and g['product_count'].std() > 0:
        sens = g['product_discount'].corr(g['product_count'])
    else:
        sens = np.nan

    qty_pct_change = g['product_count'].pct_change()
    price_pct_change = g['price_after_discount'].pct_change()
    with np.errstate(divide='ignore', invalid='ignore'):
        elasticity = (qty_pct_change / price_pct_change).replace([np.inf, -np.inf], np.nan).mean()

    return pd.Series({
        'discount_sensitivity_score': sens,
        'price_elasticity_proxy': elasticity
    })


elasticity_df = (
    df.groupby(['customer_id', 'product_id'])
      .apply(discount_and_elasticity)
      .reset_index()
)
base = base.merge(elasticity_df, on=['customer_id', 'product_id'], how='left')

# --- Timing (derived from first/last purchase) ---
base['days_since_last_purchase'] = (REFERENCE_DATE - base['last_purchase_date']).dt.days
base['days_since_first_purchase'] = (REFERENCE_DATE - base['first_purchase_date']).dt.days
base['customer_product_lifetime_days'] = (base['last_purchase_date'] - base['first_purchase_date']).dt.days


# avg / median / std of gaps between consecutive purchase dates (per unique order date)
def gap_stats(g):
    dates = g['order_date'].drop_duplicates().sort_values()
    gaps = dates.diff().dt.days.dropna()
    return pd.Series({
        'avg_days_between_purchases': gaps.mean() if len(gaps) else np.nan,
        'median_days_between_purchases': gaps.median() if len(gaps) else np.nan,
        'inter_purchase_gap_std': gaps.std(ddof=0) if len(gaps) > 1 else 0.0,
    })


gap_df = df.groupby(['customer_id', 'product_id']).apply(gap_stats).reset_index()
base = base.merge(gap_df, on=['customer_id', 'product_id'], how='left')

# ------------------------------------------------------------------
# 3. RECENT ACTIVITY  (7d / 30d / 90d / 365d windows)
# ------------------------------------------------------------------
df['days_ago'] = (REFERENCE_DATE - df['order_date']).dt.days

for window, label in [(7, '7d'), (30, '30d'), (90, '90d'), (365, '365d')]:
    recent = df[df['days_ago'] <= window]
    agg = (
        recent.groupby(['customer_id', 'product_id'])
              .agg(**{
                  f'purchase_count_{label}': ('order_id', 'nunique'),
                  f'quantity_{label}': ('product_count', 'sum'),
                  f'spend_{label}': ('spend', 'sum'),
              })
              .reset_index()
    )
    base = base.merge(agg, on=['customer_id', 'product_id'], how='left')

# customers/products with no activity in a window should be 0, not NaN
recent_cols = [c for c in base.columns if any(w in c for w in ['_7d', '_30d', '_90d', '_365d'])]
base[recent_cols] = base[recent_cols].fillna(0)

# ------------------------------------------------------------------
# 4. TREND  (30d activity relative to 90d activity)
# ------------------------------------------------------------------
base['purchase_trend_30_vs_90'] = base.apply(
    lambda r: pct_change_ratio(r['purchase_count_30d'], r['purchase_count_90d']), axis=1)
base['quantity_trend_30_vs_90'] = base.apply(
    lambda r: pct_change_ratio(r['quantity_30d'], r['quantity_90d']), axis=1)
base['spend_trend_30_vs_90'] = base.apply(
    lambda r: pct_change_ratio(r['spend_30d'], r['spend_90d']), axis=1)

# ------------------------------------------------------------------
# 5. FREQUENCY / REORDER
# ------------------------------------------------------------------
# purchases per year, normalised by how long the customer has bought this product
base['purchase_frequency'] = base['purchase_count'] / (
    (base['customer_product_lifetime_days'] / 365.0).replace(0, np.nan)
)
base['purchase_frequency'] = base['purchase_frequency'].fillna(base['purchase_count'])

# Expected days until next purchase = last purchase + typical gap
base['reorder_due_days'] = base['avg_days_between_purchases'] - base['days_since_last_purchase']

# Reorder score: higher when customer is close to / past their usual reorder point
# (simple heuristic, bounded 0-1 via a decay curve; tune as needed)
base['reorder_score'] = np.where(
    base['avg_days_between_purchases'] > 0,
    1 / (1 + np.exp((base['reorder_due_days']) / base['avg_days_between_purchases'].replace(0, np.nan))),
    np.nan
)

# How tightly individual gaps cluster around the average gap (low std = consistent)
base['reorder_consistency_score'] = 1 - (
    base['inter_purchase_gap_std'] / base['avg_days_between_purchases'].replace(0, np.nan)
)
base['reorder_consistency_score'] = base['reorder_consistency_score'].clip(lower=0, upper=1)

# Flag: customer is overdue for a reorder relative to their own normal pattern
base['churn_risk_for_product'] = (
    base['days_since_last_purchase'] > (2 * base['avg_days_between_purchases'])
).fillna(False)

# ------------------------------------------------------------------
# 6. BEHAVIOR FLAGS
# ------------------------------------------------------------------
base['is_repeat_customer_for_product'] = base['purchase_count'] > 1

# Was there a purchase of this product in the current calendar month?
current_month = REFERENCE_DATE.to_period('M')
this_month_flag = (
    df.assign(period=df['order_date'].dt.to_period('M'))
      .assign(is_this_month=lambda x: x['period'] == current_month)
      .groupby(['customer_id', 'product_id'])['is_this_month']
      .any()
      .rename('is_first_time_this_month')
      .reset_index()
)
base = base.merge(this_month_flag, on=['customer_id', 'product_id'], how='left')

# Customer's OVERALL average quantity per purchase (across all products) — needed for bulk-buyer flag
customer_overall_avg_qty = (
    df.groupby('customer_id')['product_count'].mean().rename('cust_overall_avg_qty').reset_index()
)
base = base.merge(customer_overall_avg_qty, on='customer_id', how='left')
base['is_bulk_buyer_for_product'] = base['avg_quantity'] > base['cust_overall_avg_qty']

# Gift pattern: one/two big irregular orders rather than steady small ones
base['is_gift_pattern'] = (
    (base['purchase_count'] <= 2) &
    (base['max_quantity'] > 2 * base['avg_quantity'].replace(0, np.nan))
).fillna(False)

# ------------------------------------------------------------------
# 7. SEASONALITY
# ------------------------------------------------------------------
df['order_month'] = df['order_date'].dt.month
df['order_dow'] = df['order_date'].dt.dayofweek       # 0 = Monday
df['is_weekend'] = df['order_dow'].isin([5, 6])
df['order_quarter'] = df['order_date'].dt.quarter

seasonality = (
    df.groupby(['customer_id', 'product_id'])
      .agg(
          purchase_month_mode=('order_month', mode_or_nan),
          purchase_dow_mode=('order_dow', mode_or_nan),
          is_weekend_buyer=('is_weekend', 'mean'),   # fraction of purchases on weekends
      )
      .reset_index()
)


def seasonal_index(g):
    """Share of this customer-product's purchases falling in its most common quarter."""
    counts = g['order_quarter'].value_counts(normalize=True)
    return counts.max() if len(counts) else np.nan


seas_idx = df.groupby(['customer_id', 'product_id']).apply(seasonal_index).rename('seasonal_index').reset_index()
seasonality = seasonality.merge(seas_idx, on=['customer_id', 'product_id'], how='left')
base = base.merge(seasonality, on=['customer_id', 'product_id'], how='left')

# ------------------------------------------------------------------
# 8. BASKET CONTEXT  (needs the full order, i.e. every product in each order_id)
# ------------------------------------------------------------------
order_products = df.groupby('order_id')['product_id'].apply(set)          # set of products per order
order_value = df.groupby('order_id')['spend'].sum()                       # total value of each order
order_basket_size = df.groupby('order_id')['product_id'].nunique()        # distinct products per order

df['basket_size_of_order'] = df['order_id'].map(order_basket_size)
df['order_total_value'] = df['order_id'].map(order_value)

basket_ctx = (
    df.groupby(['customer_id', 'product_id'])
      .agg(
          basket_size_avg=('basket_size_of_order', 'mean'),
          avg_order_value_when_bought=('order_total_value', 'mean'),
      )
      .reset_index()
)
base = base.merge(basket_ctx, on=['customer_id', 'product_id'], how='left')

# co_purchase_count needs a SPECIFIC "other product" to compare against.
# Below computes, for every (customer, product) pair, how many orders also
# contained that customer's single most-frequently co-purchased product.
def top_co_purchase_count(cust_id, prod_id):
    orders_with_prod = df[(df['customer_id'] == cust_id) & (df['product_id'] == prod_id)]['order_id'].unique()
    co_products = df[(df['order_id'].isin(orders_with_prod)) & (df['product_id'] != prod_id)]['product_id']
    if co_products.empty:
        return np.nan
    return co_products.value_counts().iloc[0]

base['co_purchase_count'] = base.apply(
    lambda r: top_co_purchase_count(r['customer_id'], r['product_id']), axis=1
)

# ------------------------------------------------------------------
# 9. CUSTOMER BASELINE  (one row per customer_id, then merged in)
# ------------------------------------------------------------------
customer_baseline = (
    df.groupby('customer_id')
      .agg(
          customer_avg_discount=('product_discount', 'mean'),
          customer_avg_price=('price_after_discount', 'mean'),
          customer_total_orders=('order_id', 'nunique'),
          customer_total_spend=('spend', 'sum'),
          customer_unique_products=('product_id', 'nunique'),
      )
      .reset_index()
)
customer_last_order = df.groupby('customer_id')['order_date'].max().rename('cust_last_order').reset_index()
customer_baseline = customer_baseline.merge(customer_last_order, on='customer_id')
customer_baseline['customer_recency'] = (REFERENCE_DATE - customer_baseline['cust_last_order']).dt.days

customer_first_order = df.groupby('customer_id')['order_date'].min().rename('cust_first_order').reset_index()
customer_baseline = customer_baseline.merge(customer_first_order, on='customer_id')
customer_baseline['customer_tenure_span_days'] = (REFERENCE_DATE - customer_baseline['cust_first_order']).dt.days
customer_baseline['customer_frequency'] = customer_baseline['customer_total_orders'] / (
    (customer_baseline['customer_tenure_span_days'] / 365.0).replace(0, np.nan)
)

# Loyalty score: simple weighted composite of frequency + recency (inverse) + tenure
freq_n = customer_baseline['customer_frequency'] / customer_baseline['customer_frequency'].max()
rec_n = 1 - (customer_baseline['customer_recency'] / customer_baseline['customer_recency'].max())
ten_n = customer_baseline['customer_tenure_span_days'] / customer_baseline['customer_tenure_span_days'].max()
customer_baseline['customer_loyalty_score'] = (0.4 * freq_n + 0.3 * rec_n + 0.3 * ten_n).clip(0, 1)

customer_baseline = customer_baseline.drop(columns=['cust_last_order', 'cust_first_order'])
base = base.merge(customer_baseline, on='customer_id', how='left')

# ------------------------------------------------------------------
# 10. PRODUCT BASELINE  (one row per product_id, then merged in)
# ------------------------------------------------------------------
product_baseline = (
    df.groupby('product_id')
      .agg(
          product_avg_price=('price_after_discount', 'mean'),
          product_popularity=('order_id', 'nunique'),
          product_unique_customers=('customer_id', 'nunique'),
      )
      .reset_index()
)

for window, label in [(30, '30d'), (90, '90d')]:
    sales = df[df['days_ago'] <= window].groupby('product_id')['spend'].sum().rename(f'product_sales_{label}')
    product_baseline = product_baseline.merge(sales.reset_index(), on='product_id', how='left')
product_baseline[['product_sales_30d', 'product_sales_90d']] = product_baseline[
    ['product_sales_30d', 'product_sales_90d']].fillna(0)

# growth rate: sales in most recent 30d window vs the prior 30d window (day 31-60 ago)
prior_30 = df[(df['days_ago'] > 30) & (df['days_ago'] <= 60)].groupby('product_id')['spend'].sum()
product_baseline = product_baseline.merge(prior_30.rename('prior_30d_sales').reset_index(), on='product_id', how='left')
product_baseline['prior_30d_sales'] = product_baseline['prior_30d_sales'].fillna(0)
product_baseline['product_growth_rate'] = product_baseline.apply(
    lambda r: pct_change_ratio(r['product_sales_30d'], r['prior_30d_sales']) - 1
    if r['prior_30d_sales'] else np.nan, axis=1
)
product_baseline = product_baseline.drop(columns=['prior_30d_sales'])

# concentration: % of product's total spend coming from its top 10% of customers
def concentration_top10(g):
    cust_spend = g.groupby('customer_id')['spend'].sum().sort_values(ascending=False)
    n_top = max(1, int(np.ceil(0.10 * len(cust_spend))))
    return cust_spend.iloc[:n_top].sum() / cust_spend.sum() if cust_spend.sum() else np.nan

conc = df.groupby('product_id').apply(concentration_top10).rename('product_customer_concentration').reset_index()
product_baseline = product_baseline.merge(conc, on='product_id', how='left')

# new vs returning customer ratio for the product (returning = >1 order for that product)
def new_vs_returning(g):
    order_counts = g.groupby('customer_id')['order_id'].nunique()
    returning = (order_counts > 1).sum()
    new = (order_counts == 1).sum()
    return new / returning if returning else np.nan

nvr = df.groupby('product_id').apply(new_vs_returning).rename('new_vs_returning_ratio').reset_index()
product_baseline = product_baseline.merge(nvr, on='product_id', how='left')

base = base.merge(product_baseline, on='product_id', how='left')

# ------------------------------------------------------------------
# 11. COMPARISON  (product vs customer's own normal behaviour)
# ------------------------------------------------------------------
base['price_difference_from_customer_avg'] = base['avg_price'] - base['customer_avg_price']
base['discount_difference_from_customer_avg'] = base['avg_discount'] - base['customer_avg_discount']

# ------------------------------------------------------------------
# 12. SHARE / RANKING
# ------------------------------------------------------------------
base['customer_product_share'] = base['purchase_count'] / base['customer_total_orders']
base['customer_spend_share'] = base['total_spend'] / base['customer_total_spend']
base['product_customer_share'] = base['total_spend'] / base['product_sales_30d'].replace(0, np.nan)  # see note below
# NOTE: "product's total sales" should ideally be ALL-TIME product sales, not just 30d.
# Swap in a true all-time total if you have one, e.g. product_baseline['product_total_sales_all_time'].

base['product_rank_for_customer'] = base.groupby('customer_id')['purchase_count'].rank(ascending=False, method='min')
base['product_spend_rank_for_customer'] = base.groupby('customer_id')['total_spend'].rank(ascending=False, method='min')
base['customer_rank_for_product'] = base.groupby('product_id')['total_spend'].rank(ascending=False, method='min')
base['customer_product_recency_rank'] = base.groupby('customer_id')['last_purchase_date'].rank(ascending=False, method='min')

# ------------------------------------------------------------------
# 13. GEOGRAPHY  (needs customer_df with an Address column)
# ------------------------------------------------------------------
# Uncomment and adapt once customer_df is loaded:
#
# def parse_region(address):
#     # naive example: assume "Street, City, State" format -> take the City
#     parts = str(address).split(',')
#     return parts[-2].strip() if len(parts) >= 2 else np.nan
#
# customer_df['region_from_address'] = customer_df['Address'].apply(parse_region)
# base = base.merge(customer_df[['customer_id', 'region_from_address']], on='customer_id', how='left')

# ------------------------------------------------------------------
# 14. COMPOSITE
# ------------------------------------------------------------------
# affinity_score: blend of how often, how recently, and how much of their
# wallet the customer devotes to this product (all normalised 0-1)
freq_n2 = base['purchase_count'] / base.groupby('customer_id')['purchase_count'].transform('max')
rec_n2 = 1 - (base['days_since_last_purchase'] / base.groupby('customer_id')['days_since_last_purchase'].transform('max').replace(0, np.nan))
share_n2 = base['customer_spend_share'] / base.groupby('customer_id')['customer_spend_share'].transform('max')
base['affinity_score'] = (0.4 * freq_n2 + 0.3 * rec_n2.fillna(0) + 0.3 * share_n2).clip(0, 1)

# customer_tenure_days needs a signup_date column from customer_df:
#
# customer_df['customer_tenure_days'] = (REFERENCE_DATE - pd.to_datetime(customer_df['signup_date'])).dt.days
# base = base.merge(customer_df[['customer_id', 'customer_tenure_days']], on='customer_id', how='left')

# ------------------------------------------------------------------
# DONE
# ------------------------------------------------------------------
print(f"Final feature table shape: {base.shape}")
base.sample(5)

# base.to_csv("customer_product_features.csv", index=False)



In [ ]:
base.columns